# 04 - Scorecard

Notebook para converter o modelo em score, analisar faixas de risco e documentar regras de decisao.

## 1 — Setup e carregamento dos artefatos

Este notebook é **self-contained**: parte dos artefatos gerados pelo notebook 03 (`modelo_credito.pkl`, `imputers.pkl`, `encoders.pkl` em `../models/`). Execute o notebook 03 antes deste para gerá-los.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTENC

from src.preprocessing import tratar_nulos
from src.features import criar_features, codificar_categoricas
from src.scoring import (
    adjust_probability_to_prior,
    probability_to_scorecard_points,
    assign_risk_band,
)

# Artefatos gerados pelo notebook 03 (execute-o antes deste)
modelo_final = joblib.load('../models/modelo_credito.pkl')
imputers = joblib.load('../models/imputers.pkl')
encoders = joblib.load('../models/encoders.pkl')
print('Artefatos carregados:', type(modelo_final).__name__)

## 2 — Reconstrói treino/teste (mesmo split do notebook 03)

Reaplica o mesmo split (`random_state=42`) e o pré-processamento aprendido no treino — carregado dos artefatos, com `fit=False` — garantindo consistência total com o modelo treinado.

In [ ]:
df = pd.read_csv('../data/raw/application_train.csv')
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

# Mesmo split do notebook 03 (random_state=42, stratify) -> consistência total
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Reaplica o pré-processamento aprendido no treino (fit=False -> sem leakage)
X_train = tratar_nulos(X_train, fit=False, imputers=imputers)
X_test = tratar_nulos(X_test, fit=False, imputers=imputers)
X_train = criar_features(X_train)
X_test = criar_features(X_test)
X_train = codificar_categoricas(X_train, fit=False, encoders=encoders)
X_test = codificar_categoricas(X_test, fit=False, encoders=encoders)

# Treino balanceado (SMOTENC), igual ao notebook 03
cat_idx = [X_train.columns.get_loc(col) for col in encoders]
X_train_bal, y_train_bal = SMOTENC(
    categorical_features=cat_idx, random_state=42, k_neighbors=5
).fit_resample(X_train, y_train)

print('Treino balanceado:', X_train_bal.shape, '| Teste:', X_test.shape)

## 3 — Seleção de features (SHAP) + Regressão Logística

O scorecard usa **Regressão Logística** (modelo linear, transparente e bem aceito por reguladores) sobre as 10 features mais importantes segundo o SHAP do modelo campeão (LightGBM).

In [ ]:
# Top 10 features pelo SHAP (recalculado a partir do modelo carregado)
explainer = shap.TreeExplainer(modelo_final)
X_sample = X_test.sample(1000, random_state=42)
shap_values = explainer.shap_values(X_sample)
shap_vals_inadim = shap_values[1] if isinstance(shap_values, list) else shap_values

shap_importance = pd.DataFrame({
    'feature': X_sample.columns,
    'importance': np.abs(shap_vals_inadim).mean(axis=0)
}).sort_values('importance', ascending=False)

top_features = shap_importance.head(10)['feature'].tolist()
print('Top 10 features para o scorecard:')
print(top_features)

# Regressão Logística nas top features (escalonadas)
scaler = StandardScaler()
X_sc_train_scaled = scaler.fit_transform(X_train_bal[top_features])
X_sc_test_scaled = scaler.transform(X_test[top_features])

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_sc_train_scaled, y_train_bal)

auc_lr = roc_auc_score(y_test, lr.predict_proba(X_sc_test_scaled)[:, 1])
print(f'AUC Regressão Logística: {auc_lr:.4f}')

## 4 — Conversão para pontos do scorecard (PDO)

Duas etapas de calibração antes de pontuar:

1. **Correção de prior** (`adjust_probability_to_prior`): a LR foi treinada em dados balanceados (SMOTE 50/50), então suas probabilidades são infladas (~0.40 em vez de ~0.08). Corrigi os log-odds para o prior real da população, recuperando PDs verdadeiras.
2. **Conversão PDO/log-odds** (`probability_to_scorecard_points`): `odds_ref` = odds good/bad da população (centra o score) e `PDO=60` para um spread legível em 300–850.

In [ ]:
# 1) Corrige o viés de probabilidade do treino balanceado para o prior real
prior_real = y_train.mean()  # taxa de inadimplência real (~0.08)
prob_lr = lr.predict_proba(X_sc_test_scaled)[:, 1]
prob_real = adjust_probability_to_prior(prob_lr, target_prior=prior_real, source_prior=0.5)

# 2) Converte as PDs corrigidas em pontos (PDO). odds_ref = odds good/bad da população
odds_ref = (1 - prior_real) / prior_real
scores = probability_to_scorecard_points(prob_real, pdo=60, odds_ref=odds_ref, score_ref=600)

print(f'Prior real: {prior_real:.4f} | PD média corrigida: {prob_real.mean():.4f}')
print(f'Score mínimo: {scores.min()} | médio: {scores.mean():.0f} | máximo: {scores.max()}')

# Distribuição dos scores por classe (bons pagadores devem ter score mais alto)
df_scores = pd.DataFrame({'score': scores, 'target': y_test.values})
df_scores.groupby('target')['score'].describe()

## 5 — Faixas de risco e regra de decisão

As faixas são definidas por **tercis do score** (data-driven), garantindo populações equilibradas em `low`/`medium`/`high`. A validação esperada: a inadimplência observada cresce de `low` → `medium` → `high`.

In [ ]:
# Faixas por tercil do score (data-driven) — popula low/medium/high de forma equilibrada
c_high, c_low = np.quantile(df_scores['score'], [1/3, 2/3])
df_scores['faixa'] = df_scores['score'].apply(
    lambda s: assign_risk_band(s, medium_min=c_high, low_min=c_low)
)
print(f'Cortes: high < {c_high:.0f} <= medium < {c_low:.0f} <= low')

# Taxa de inadimplência observada por faixa (deve crescer de low -> high)
resumo = df_scores.groupby('faixa').agg(
    qtd=('target', 'size'),
    inadimplencia=('target', 'mean'),
    score_medio=('score', 'mean'),
).reindex(['low', 'medium', 'high'])
print(resumo)

plt.figure(figsize=(8, 5))
for t, g in df_scores.groupby('target'):
    plt.hist(g['score'], bins=40, alpha=0.6, label=f'TARGET={t}')
for corte in (c_high, c_low):
    plt.axvline(corte, color='gray', linestyle='--', linewidth=1)
plt.xlabel('Score')
plt.ylabel('Frequência')
plt.title('Distribuição do score por classe (cortes das faixas)', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/scorecard_distribuicao.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — Conclusão e interpretação de negócio

O scorecard traduz o modelo em uma escala de 300–850 acionável, validada no conjunto de teste completo (61.503 clientes). As faixas separam o risco de forma clara e monotônica:

| Faixa | População | Inadimplência observada | Leitura de negócio |
|---|---|---|---|
| **low** (score alto) | ~1/3 | **~3%** | Risco baixo — aprovação automática, melhores condições/limite |
| **medium** | ~1/3 | **~6%** | Risco intermediário — aprovar com análise, limite/juros ajustados |
| **high** (score baixo) | ~1/3 | **~15%** | Risco alto — revisão manual, garantias ou recusa |

**Como vira política de crédito:** os cortes de tercil são um ponto de partida. Na prática, eles seriam ajustados ao **apetite de risco** e à economia da carteira (ex.: mover o corte da faixa `low` para baixar a inadimplência aprovada, ou usar a PD corrigida diretamente em pricing baseado em risco). A faixa `high` concentra ~5x mais inadimplência que a `low` — é onde a triagem evita a maior parte das perdas.

**Pontos de atenção (honestidade técnica):**
- As probabilidades passam por **correção de prior** (treino balanceado → população real ~8%), então a PD é interpretável como taxa esperada de inadimplência.
- Os cortes de faixa são **data-driven** (tercis); recalibrar periodicamente conforme a distribuição muda (drift) — vide notebook 05 (monitoramento).
- Parte das features de Open Finance é **simulada** (`criar_features_open_finance_simulado`); num cenário real, substituir por dados transacionais efetivos tende a elevar o poder discriminante.
- O scorecard usa Regressão Logística (interpretável, exigência regulatória); o LightGBM permanece como modelo ideal para benchmark e seleção de features (SHAP).